# 02 - Entrenamiento local de los 4 modelos (prueba con muestra)

Corre los 4 modelos del plan **uno a la vez**, con una muestra reducida y ESTRATIFICADA (por año y por clase del target) de `dataset_modelado_personas.csv`, para probar que todo el pipeline (Time Series Split + SMOTE + cada arquitectura) funciona sin saturar una laptop de capacidad media, antes de correr con el dataset completo.

**Cómo usar este notebook:** ejecuta una sección de modelo a la vez. Después de cada una, revisa la tabla de resultados y el tiempo de entrenamiento impresos antes de pasar a la siguiente. Si tu equipo se satura durante un modelo, interrumpe el kernel (botón de stop) antes de continuar -- no hace falta reiniciar todo el notebook, solo esa celda.

Los modelos están ordenados del más liviano al más pesado: XGBoost -> LightGBM -> CNN-LSTM -> TabNet.

In [1]:
%run ./00_parametros_globales.ipynb

import sys
import pandas as pd

sys.path.insert(0, str(RAIZ_PROYECTO / "src" / "features"))
sys.path.insert(0, str(RAIZ_PROYECTO / "src" / "evaluation"))
sys.path.insert(0, str(RAIZ_PROYECTO / "src" / "models"))

from preparacion_modelado import muestrear_estratificado

c:\TesisSD\.venv\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


GPU NVIDIA detectada: False
  Corriendo en CPU. Se recomienda SAMPLING_MODE=True para iterar más rápido.

RAIZ_PROYECTO: c:\TesisSD
SAMPLING_MODE: False (fracción=1.0)
VENTANA_TEMPORAL_ANIOS: 3
RUTA_DATASET_PERSONAS existe: True
RUTA_PANEL_MACRO existe: True


## 0. Verificación del entorno

Antes de correr cualquier modelo, confirma que las librerías necesarias están instaladas (`pip install -r requirements.txt` en tu entorno). XGBoost y LightGBM son livianas; CNN-LSTM y TabNet dependen de `torch` (y TabNet además de `pytorch-tabnet`), que son más pesadas de instalar -- mejor detectarlo aquí que a mitad de un entrenamiento.

In [2]:
paquetes = ["xgboost", "lightgbm", "torch", "pytorch_tabnet", "imblearn", "sklearn"]
for paquete in paquetes:
    try:
        __import__(paquete)
        print(f"OK      {paquete}")
    except ImportError as e:
        print(f"FALTA   {paquete}  ({e})")

OK      xgboost
OK      lightgbm
OK      torch
OK      pytorch_tabnet
OK      imblearn
OK      sklearn


## 1. Cargar datos y tomar la muestra estratificada

`panel_macro_anual.csv` NO se muestrea (es un panel anual pequeño, una fila por año 2007-2024; muestrearlo rompería la ventana temporal del CNN-LSTM). Solo se reduce `dataset_modelado_personas.csv`, que es el que tiene un tamaño relevante para el tiempo de entrenamiento.

In [3]:
df_completo = pd.read_csv(RUTA_DATASET_PERSONAS)
panel_macro = pd.read_csv(RUTA_PANEL_MACRO)

if SAMPLING_MODE:
    df_muestra = muestrear_estratificado(df_completo, fraccion=FRACCION_MUESTRA, random_state=RANDOM_STATE)
else:
    df_muestra = df_completo

print(f"Filas dataset completo: {len(df_completo)}")
print(f"Filas de la muestra usada en este notebook: {len(df_muestra)} (SAMPLING_MODE={SAMPLING_MODE})")
print(f"\nAños presentes en la muestra: {sorted(df_muestra['anio'].unique())}")
print("\nBalance de la muestra por año (%, clase 'Satisfecho'):")
print((100 * df_muestra.dropna(subset=["satisfecho_democracia"]).groupby("anio")["satisfecho_democracia"].mean()).round(1))

Filas dataset completo: 15600
Filas de la muestra usada en este notebook: 15600 (SAMPLING_MODE=False)

Años presentes en la muestra: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2013), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020), np.int64(2023), np.int64(2024)]

Balance de la muestra por año (%, clase 'Satisfecho'):
anio
2007    36.0
2008    39.5
2009    36.0
2010    50.4
2011    49.8
2013    61.2
2015    59.6
2016    42.0
2017    51.6
2018    36.5
2020    10.1
2023    12.0
2024    19.1
Name: satisfecho_democracia, dtype: float64


## 2. Modelo 1/4 -- XGBoost

El más rápido de los 4; recomendado para probar primero que el pipeline completo (Time Series Split + SMOTE por fold) funciona bien con tu instalación.

In [4]:
import baseline_xgboost

resultados_xgb, modelos_xgb = baseline_xgboost.entrenar_evaluar_xgboost(df_muestra, min_anios_train=5)
baseline_xgboost.guardar_resultados(resultados_xgb, out_path=RUTA_TABLAS / "resultados_baseline_xgboost.csv")

[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 5695  ->  después: 6554
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTE va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Fila

### SHAP (TreeSHAP) -- XGBoost

TreeSHAP (`shap.TreeExplainer`) es EXACTO para modelos de árboles -- no necesita background ni muestreo (a diferencia de KernelSHAP en CNN-LSTM/TabNet), así que se calcula sobre el fold de prueba COMPLETO de cada año. Reutiliza `modelos_xgb` (ya entrenado arriba).

In [5]:
from shap_explicabilidad import explicar_arbol, guardar_resumen_shap, graficar_resumen_shap

explicaciones_xgb = explicar_arbol(df_muestra if SAMPLING_MODE else df_completo, modelos_xgb, "xgboost")
for anio, r in explicaciones_xgb.items():
    print(f"--- fold {anio} ---")
    print(r["resumen_global"])
    guardar_resumen_shap(r["resumen_global"], RUTA_TABLAS / f"shap_global_xgboost_{anio}.csv")
    graficar_resumen_shap(r["resumen_global"], f"XGBoost -- importancia SHAP global ({anio})", RUTA_FIGURES / f"shap_global_xgboost_{anio}.png")

[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 5695  ->  después: 6554
[xgboost] SHAP calculado para fold de prueba 2013 (1161 encuestados reales, no sintéticos).
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTE va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat

### Ejemplo de explicación LOCAL (auditoría de un encuestado puntual)

A diferencia del resumen global (impacto macro agregado), la explicación LOCAL muestra qué variables empujaron la predicción de UN encuestado en particular -- útil para auditar un perfil sociodemográfico o año específico, como pide la propuesta. Ejemplo con el primer encuestado del fold más reciente de XGBoost; cambia `idx` o `ultimo_anio_xgb` para auditar otro encuestado/año.

In [6]:
from shap_explicabilidad import explicar_perfil_local
from config_features import FEATURES_CATEGORICAS, FEATURES_NUMERICAS

ultimo_anio_xgb = max(explicaciones_xgb.keys())
perfil_local = explicar_perfil_local(
    explicaciones_xgb[ultimo_anio_xgb]["shap_values"],
    explicaciones_xgb[ultimo_anio_xgb]["X_test"],
    FEATURES_CATEGORICAS + FEATURES_NUMERICAS,
    idx=0,
)
print(f"Explicación LOCAL del primer encuestado del fold {ultimo_anio_xgb} (XGBoost):")
print(perfil_local.to_string(index=False))

Explicación LOCAL del primer encuestado del fold 2024 (XGBoost):
                          feature     valor_observado  shap_value
                    empleo_formal           41.679028   -0.602995
                   democ_supp_cat prefiere_democracia    0.279801
         confidence_congress_alta                 0.0   -0.153476
        confidence_judiciary_alta                 0.0   -0.120708
                      job_concern                 1.0   -0.112788
             confidence_army_alta                 1.0    0.109548
                    v2x_polyarchy               0.651   -0.094598
                   v2xeg_eqprotec               0.255   -0.092792
confidence_political_parties_alta                 0.0   -0.083183
                     v2x_delibdem                0.35   -0.061516


## 3. Modelo 2/4 -- LightGBM

Corre esta celda solo después de revisar los resultados de XGBoost arriba.

In [7]:
import baseline_lightgbm

resultados_lgbm, modelos_lgbm = baseline_lightgbm.entrenar_evaluar_lightgbm(df_muestra, min_anios_train=5)
baseline_lightgbm.guardar_resultados(resultados_lgbm, out_path=RUTA_TABLAS / "resultados_baseline_lightgbm.csv")

[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 5695  ->  después: 6554
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTE va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Fila

### SHAP (TreeSHAP) -- LightGBM

Igual que XGBoost: TreeSHAP es exacto (no necesita background ni muestreo), así que se calcula sobre el fold de prueba COMPLETO de cada año, sin ningún recorte.

In [8]:
from shap_explicabilidad import explicar_arbol, guardar_resumen_shap, graficar_resumen_shap

explicaciones_lgbm = explicar_arbol(df_muestra if SAMPLING_MODE else df_completo, modelos_lgbm, "lightgbm")
for anio, r in explicaciones_lgbm.items():
    print(f"--- fold {anio} ---")
    print(r["resumen_global"])
    guardar_resumen_shap(r["resumen_global"], RUTA_TABLAS / f"shap_global_lightgbm_{anio}.csv")
    graficar_resumen_shap(r["resumen_global"], f"LightGBM -- importancia SHAP global ({anio})", RUTA_FIGURES / f"shap_global_lightgbm_{anio}.png")

[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 5695  ->  después: 6554


c:\TesisSD\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[lightgbm] SHAP calculado para fold de prueba 2013 (1161 encuestados reales, no sintéticos).
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTE va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 6856  ->  después: 7454


c:\TesisSD\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[lightgbm] SHAP calculado para fold de prueba 2015 (1174 encuestados reales, no sintéticos).
Balance de este fold -- clase 1 (Satisfecho): 47.68%, clase 0 (No Satisfecho): 52.32%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 47.7%). SMOTE va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 8030  ->  después: 8402


...

[lightgbm] SHAP calculado para fold de prueba 2020 (1164 encuestados reales, no sintéticos).
Balance de este fold -- clase 1 (Satisfecho): 43.07%, clase 0 (No Satisfecho): 56.93%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 12717  ->  después: 14480


c:\TesisSD\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[lightgbm] SHAP calculado para fold de prueba 2023 (1186 encuestados reales, no sintéticos).
Balance de este fold -- clase 1 (Satisfecho): 40.42%, clase 0 (No Satisfecho): 59.58%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 13903  ->  después: 16568


c:\TesisSD\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[lightgbm] SHAP calculado para fold de prueba 2024 (1196 encuestados reales, no sintéticos).
--- fold 2013 ---
                              feature  importancia_media_abs_shap
0                  econ_situation_cat                    0.504691
1                      democ_supp_cat                    0.377420
2            confidence_congress_alta                    0.331387
3        resp_economic_perception_cat                    0.306081
4   confidence_political_parties_alta                    0.182545
5              confidence_police_alta                    0.151758
6                      tasa_desempleo                    0.137903
7                            resp_age                    0.136680
8                confidence_army_alta                    0.125381
9           tasa_participacion_global                    0.122494
10                        job_concern                    0.120135
11          confidence_judiciary_alta                    0.112414
12                      ideolog

## 4. Modelo 3/4 -- CNN-LSTM

Requiere `torch` instalado (ver verificación del entorno arriba). Es notablemente más pesado que los 2 anteriores: el número de épocas ahora se ajusta SOLO según `SAMPLING_MODE` (5 épocas en modo prueba, 15 -- el valor por defecto ya optimizado del script, ver docstring de `entrenar_evaluar_cnn_lstm` -- en la corrida real), no hace falta editar nada a mano en la celda de abajo.

In [9]:
import cnn_lstm

resultados_cnn, modelos_cnn = cnn_lstm.entrenar_evaluar_cnn_lstm(
    df_muestra, panel_macro,
    n_epochs=5 if SAMPLING_MODE else 15,  # 15 = valor por defecto ya optimizado del script (ver docstring)
)
cnn_lstm.guardar_resultados(resultados_cnn, out_path=RUTA_TABLAS / "resultados_cnn_lstm.csv")

[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 5695  ->  después: 6554
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTE va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Fila

In [10]:
from shap_explicabilidad import explicar_cnn_lstm, guardar_resumen_shap, graficar_resumen_shap

explicaciones_cnn = explicar_cnn_lstm(df_muestra if SAMPLING_MODE else df_completo, panel_macro, modelos_cnn)
for anio, r in explicaciones_cnn.items():
    print(f"--- fold {anio} ---")
    print(r["resumen_global"])
    guardar_resumen_shap(r["resumen_global"], RUTA_TABLAS / f"shap_global_cnn_lstm_{anio}.csv")
    graficar_resumen_shap(r["resumen_global"], f"CNN-LSTM -- importancia SHAP global ({anio})", RUTA_FIGURES / f"shap_global_cnn_lstm_{anio}.png")

[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 5695  ->  después: 6554
[cnn_lstm] SHAP (KernelSHAP) calculado para fold de prueba 2013 (300 encuestados reales).
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTE va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat',

## 5. Modelo 4/4 -- TabNet

Requiere `torch` + `pytorch-tabnet`. Igual que el CNN-LSTM, el número de épocas se ajusta SOLO según `SAMPLING_MODE` (10 en modo prueba, 100 -- el valor por defecto del script -- en la corrida real).

In [11]:
import tabnet_model

resultados_tabnet, modelos_tabnet = tabnet_model.entrenar_evaluar_tabnet(
    df_muestra,
    max_epochs=10 if SAMPLING_MODE else 100,  # 100 = valor por defecto del script
)
tabnet_model.guardar_resultados(resultados_tabnet, out_path=RUTA_TABLAS / "resultados_tabnet.csv")

[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 5695  ->  después: 6554


c:\TesisSD\.venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTE va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 6856  ->  después: 7454


c:\TesisSD\.venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


Balance de este fold -- clase 1 (Satisfecho): 47.68%, clase 0 (No Satisfecho): 52.32%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 47.7%). SMOTE va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 8030  ->  después: 8402


...

Balance de este fold -- clase 1 (Satisfecho): 43.07%, clase 0 (No Satisfecho): 56.93%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 12717  ->  después: 14480


c:\TesisSD\.venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


Balance de este fold -- clase 1 (Satisfecho): 40.42%, clase 0 (No Satisfecho): 59.58%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 13903  ->  después: 16568


c:\TesisSD\.venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)



[tabnet_model] Resultados por fold:
 anio_test  n_anios_train  n_train  n_test  accuracy  f1_macro  f1_weighted  f1_satisfecho  pr_auc  tiempo_entrenamiento_seg
      2013              5     5695    1161    0.6253    0.6177       0.6298         0.6717  0.7289                    105.59
      2015              6     6856    1174    0.6457    0.6124       0.6342         0.7260  0.7456                    116.59
      2016              7     8030    1174    0.6014    0.6013       0.6008         0.6047  0.5853                    116.96
      2017              8     9204    1183    0.5866    0.5861       0.5856         0.5707  0.6502                    130.41
      2018              9    10387    1166    0.5823    0.5640       0.5881         0.4746  0.4640                    155.42
      2020             10    11553    1164    0.7491    0.5557       0.7894         0.2626  0.2356                    179.17
      2023             11    12717    1186    0.8803    0.4682       0.8242         0.00

In [12]:
from shap_explicabilidad import explicar_tabnet

explicaciones_tabnet = explicar_tabnet(df_muestra if SAMPLING_MODE else df_completo, modelos_tabnet)
for anio, r in explicaciones_tabnet.items():
    print(f"--- fold {anio} ---")
    print(r["resumen_global"])
    guardar_resumen_shap(r["resumen_global"], RUTA_TABLAS / f"shap_global_tabnet_{anio}.csv")
    graficar_resumen_shap(r["resumen_global"], f"TabNet -- importancia SHAP global ({anio})", RUTA_FIGURES / f"shap_global_tabnet_{anio}.png")

[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 5695  ->  después: 6554
[tabnet] SHAP (KernelSHAP) calculado para fold de prueba 2013 (300 encuestados reales).
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTE va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', '

## 6. Extensión: variables macro REZAGADAS (a pedido del tutor)

El tutor pidió reforzar la componente **temporal** también en los modelos no secuenciales (XGBoost, LightGBM, TabNet), agregando rezagos del estilo `empleo_t | empleo_t-1 | empleo_t-3`.

`src/features/rezagos_macro.py` agrega columnas `{variable}_lag1` y `{variable}_lag3` para 6 indicadores clave (`tasa_desempleo`, `gini_ingpc`, `ingreso_promedio_pc`, `pobreza_ingresos`, `v2x_polyarchy`, `v2x_libdem`), tomadas de `panel_macro_anual.csv` (ENEMDU/V-Dem, serie anual SIN huecos -- por eso los huecos de Latinobarómetro no afectan a estos rezagos).

Abajo se comparan los resultados de XGBoost **con** y **sin** estos rezagos, sobre la misma muestra estratificada ya cargada arriba.

In [15]:
from rezagos_macro import agregar_rezagos_a_dataset

# panel_macro ya se cargó en la Sección 1 -- se reutiliza, no hace falta releerlo.
df_muestra_ext, columnas_lag = agregar_rezagos_a_dataset(df_muestra, panel_macro)
print(f"Columnas de rezago agregadas: {columnas_lag}")
print(f"Shape muestra: {df_muestra.shape}  ->  con rezagos: {df_muestra_ext.shape}")

Columnas de rezago agregadas: ['tasa_desempleo_lag1', 'tasa_desempleo_lag3', 'gini_ingpc_lag1', 'gini_ingpc_lag3', 'ingreso_promedio_pc_lag1', 'ingreso_promedio_pc_lag3', 'pobreza_ingresos_lag1', 'pobreza_ingresos_lag3', 'v2x_polyarchy_lag1', 'v2x_polyarchy_lag3', 'v2x_libdem_lag1', 'v2x_libdem_lag3']
Shape muestra: (15600, 71)  ->  con rezagos: (15600, 83)


In [14]:
import baseline_xgboost
import importlib
importlib.reload(baseline_xgboost)

# Sin rezagos (features_numericas por defecto, ya corrido en la Sección 2 -- se repite aquí para comparar lado a lado)
resultados_sin_lags, _ = baseline_xgboost.entrenar_evaluar_xgboost(df_muestra, min_anios_train=5, random_state=RANDOM_STATE)

# Con rezagos: FEATURES_NUMERICAS + columnas_lag
from config_features import FEATURES_NUMERICAS
resultados_con_lags, _ = baseline_xgboost.entrenar_evaluar_xgboost(
    df_muestra_ext, features_numericas=FEATURES_NUMERICAS + columnas_lag, min_anios_train=5, random_state=RANDOM_STATE
)

print(f"PR-AUC promedio SIN rezagos: {resultados_sin_lags['pr_auc'].mean():.4f}")
print(f"PR-AUC promedio CON rezagos: {resultados_con_lags['pr_auc'].mean():.4f}")

[preparacion_modelado] 15099 filas con target válido, 9 features categóricas + 39 numéricas.
Balance de este fold -- clase 1 (Satisfecho): 42.46%, clase 0 (No Satisfecho): 57.54%. Clase minoritaria detectada: Satisfecho.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Filas antes de SMOTE: 5695  ->  después: 6554
Balance de este fold -- clase 1 (Satisfecho): 45.64%, clase 0 (No Satisfecho): 54.36%. Clase minoritaria detectada: Satisfecho.
AVISO: este fold está casi balanceado (clase minoritaria = 45.6%). SMOTE va a generar pocas muestras sintéticas; considera si de verdad hace falta en este fold.
Columnas categóricas detectadas (usando SMOTENC): ['resp_sex', 'resp_education', 'resp_employment', 'resp_religion', 'democ_supp_cat', 'ideologia_cat', 'econ_situation_cat', 'resp_economic_perception_cat', 'resp_chief']
Fila

**Nota:** esta comparación corre sobre la muestra reducida (`SAMPLING_MODE=True`), pensada solo para probar que el pipeline funciona sin saturar el equipo local. La diferencia entre PR-AUC con/sin rezagos en esta muestra pequeña **no es concluyente** -- conviene repetir esta comparación con el dataset completo antes de sacar conclusiones para la tesis.

## Próximos pasos

Si los 4 modelos corrieron sin error con la muestra:

1. Cambia `SAMPLING_MODE = False` en `00_parametros_globales.ipynb`.
2. Vuelve a correr este notebook desde el inicio -- la celda de la sección 1 usará automáticamente `df_completo` en vez de la muestra, y las secciones de CNN-LSTM/TabNet ya ajustan `n_epochs`/`max_epochs` solas según `SAMPLING_MODE` (no hace falta editar esas celdas a mano).
3. Guarda esa corrida final -- es la que alimenta la tabla comparativa de la tesis (`src/evaluation/metricas.py`).